In [ ]:
import os
from google.colab import userdata

# 1. Securely fetch credentials from Colab's encrypted vault
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

# 2. Create the hidden Kaggle directory Colab expects
os.makedirs('/root/.kaggle', exist_ok=True)

# 3. Create the configuration file programmatically
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

# 4. Give the file the correct security permissions
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# 5. Download the CK+ Dataset from Kaggle
print("Credentials configured! Downloading CK+ Dataset...")
!kaggle datasets download -d shawon10/ckplus

# 6. Unzip the dataset into a folder named CK_Dataset
print("Unzipping dataset...")
!unzip -q ckplus.zip -d CK_Dataset

# 7. Print out the folders to prove it worked!
print("\nDownload Complete! Here are the emotion folders we got:")
print(os.listdir('CK_Dataset/CK+48'))

In [ ]:
import os
import random
import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class PairedFERDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.emotions = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
        self.data = []

        print("Scanning dataset for identities and emotions...")
        for label, emotion in enumerate(self.emotions):
            folder_path = os.path.join(root_dir, emotion)

            # Find all images in this emotion folder
            for img_path in glob.glob(os.path.join(folder_path, '*.*')):
                filename = os.path.basename(img_path)

                # CK+ filenames look like S010_004_00000015.png
                # We split by '_' and take the first part to get the Subject ID
                identity = filename.split('_')[0]

                self.data.append({
                    'path': img_path,
                    'emotion_idx': label,
                    'emotion_name': emotion,
                    'identity': identity
                })

        print(f"Total images found: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img1_info = self.data[idx]

        # Find a matching image: SAME emotion, DIFFERENT identity
        while True:
            img2_info = random.choice(self.data)
            same_emotion = (img2_info['emotion_idx'] == img1_info['emotion_idx'])
            different_identity = (img2_info['identity'] != img1_info['identity'])

            if same_emotion and different_identity:
                break

        # Load the images and convert to Grayscale ('L')
        img1 = Image.open(img1_info['path']).convert('L')
        img2 = Image.open(img2_info['path']).convert('L')

        # Apply transformations (Resize and convert to PyTorch Tensor)
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, img1_info['emotion_idx'], img1_info['emotion_name'], img1_info['identity'], img2_info['identity']

# --- Testing the Data Loader ---

# 1. Define transformations (Resize to 112x112 as per the research paper)
transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor()
])

# 2. Instantiate the dataset
dataset = PairedFERDataset(root_dir='CK_Dataset/CK+48', transform=transform)

# 3. Fetch one pair to prove it works!
img1, img2, em_idx, em_name, id1, id2 = dataset[0]

print("\n" + "="*50)
print(f"SUCCESS! Created a training pair for: {em_name.upper()}")
print(f"Image 1 -> Identity: {id1} | Tensor Shape: {list(img1.shape)}")
print(f"Image 2 -> Identity: {id2} | Tensor Shape: {list(img2.shape)}")
print("="*50)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# ---------------------------------------------------------
# 1. ENCODERS (ResNet-18 modified for 64-d output & Grayscale)
# ---------------------------------------------------------
class ResNet18Encoder(nn.Module):
    def __init__(self, out_dim=64):
        super(ResNet18Encoder, self).__init__()
        # Load a standard ResNet-18
        resnet = models.resnet18(weights=None)

        # Paper Sec 5.2: Images are resized to 112x112x1 (Grayscale)
        # We must modify the first convolutional layer to accept 1 channel instead of 3
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3

        # Paper Fig 2a: Expression feature map F(M)
        self.layer4 = resnet.layer4
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Paper Sec 5.3: Learn a 64-dimensional representation
        self.fc = nn.Linear(resnet.fc.in_features, out_dim)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        feature_map = self.layer4(x) # Expression feature map F(M)

        pooled = self.avgpool(feature_map)
        img_features = torch.flatten(pooled, 1) # Global 512-d feature before final reduction

        out_representation = self.fc(img_features) # Extract the final 64-d representation
        return feature_map, img_features, out_representation

# ---------------------------------------------------------
# 2. STATISTICS NETWORKS (For Mutual Information Estimation)
# ---------------------------------------------------------
class GlobalStatisticsNetwork(nn.Module):
    def __init__(self, img_feat_dim=512, z_dim=64):
        super(GlobalStatisticsNetwork, self).__init__()
        # Evaluates global U_theta(M, Z) for the Donsker-Varadhan representation (Eq. 3 & Eq. 4)
        self.net = nn.Sequential(
            nn.Linear(img_feat_dim + z_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, img_feat, z):
        # Concatenate the global image feature and the latent representation
        x = torch.cat((img_feat, z), dim=1)
        return self.net(x)

class LocalStatisticsNetwork(nn.Module):
    def __init__(self, feature_map_channels=512, z_dim=64):
        super(LocalStatisticsNetwork, self).__init__()
        # Evaluates local U_phi(F(M), Z) for Eq. 5
        # Uses 1x1 convolutions to evaluate MI across spatial locations
        self.net = nn.Sequential(
            nn.Conv2d(feature_map_channels + z_dim, 512, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(512, 256, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(256, 1, kernel_size=1)
        )

    def forward(self, feature_map, z):
        # feature_map shape: [B, 512, H, W]
        # z shape: [B, 64]
        B, C, H, W = feature_map.size()

        # Expand z spatially to match the feature map
        z_expanded = z.unsqueeze(2).unsqueeze(3).expand(B, -1, H, W)

        # Concatenate along the channel dimension
        x = torch.cat((feature_map, z_expanded), dim=1)

        # Return local score map
        return self.net(x)

# ---------------------------------------------------------
# 3. DISCRIMINATOR (To decouple Identity and Expression)
# ---------------------------------------------------------
class AdversarialDiscriminator(nn.Module):
    def __init__(self, input_dim=128): # 64-d Expression + 64-d Identity = 128
        super(AdversarialDiscriminator, self).__init__()
        # Paper Sec 5.3: "The discriminator is designed as a network comprising three fully connected layers."
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid() # Outputs a probability (0 to 1) for real vs fake pairs
        )

    def forward(self, expr_rep, id_rep):
        x = torch.cat((expr_rep, id_rep), dim=1)
        return self.net(x)

# ---------------------------------------------------------
# 4. FER CLASSIFIER (To predict the final emotion)
# ---------------------------------------------------------
class FERClassifier(nn.Module):
    def __init__(self, input_dim=64, num_classes=7):
        super(FERClassifier, self).__init__()
        # Paper Sec 5.3: "For classifier, two fully-connected hidden layers are used."
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# --- Testing the Architectures ---
print("Initializing Networks...")
encoder_exp = ResNet18Encoder()
encoder_id = ResNet18Encoder()
global_stat_net = GlobalStatisticsNetwork()
local_stat_net = LocalStatisticsNetwork()
discriminator = AdversarialDiscriminator()
classifier = FERClassifier(num_classes=7) # CK+ has 7 base classes

# Create a dummy batch of 2 grayscale images (Batch Size, Channels, Height, Width)
dummy_img = torch.randn(2, 1, 112, 112)

print("\nRunning Dummy Data through Encoders...")
feat_map_exp, img_feat, exp_vec = encoder_exp(dummy_img)
_, _, id_vec = encoder_id(dummy_img)

print(f"Expression Feature Map Shape: {list(feat_map_exp.shape)} (Expected: [2, 512, 4, 4])")
print(f"Expression Vector Shape: {list(exp_vec.shape)} (Expected: [2, 64])")
print(f"Identity Vector Shape: {list(id_vec.shape)} (Expected: [2, 64])")

print("\nRunning Dummy Data through Statistics Networks (MINE)...")
global_mi_score = global_stat_net(img_feat, exp_vec)
local_mi_score = local_stat_net(feat_map_exp, exp_vec)
print(f"Global MI Score Shape: {list(global_mi_score.shape)} (Expected: [2, 1])")
print(f"Local MI Score Map Shape: {list(local_mi_score.shape)} (Expected: [2, 1, 4, 4])")

print("\nRunning Dummy Data through Discriminator...")
disc_score = discriminator(exp_vec, id_vec)
print(f"Discriminator Output Shape: {list(disc_score.shape)} (Expected: [2, 1])")

print("\nRunning Dummy Data through Classifier...")
class_preds = classifier(exp_vec)
print(f"Classifier Predictions Shape: {list(class_preds.shape)} (Expected: [2, 7])")
print("\nSUCCESS! All networks strictly align with the research paper specifications.")

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
import math

# 1. Setup Device (Move everything to the T4 GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Move all networks to GPU
encoder_exp.to(device)
encoder_id.to(device)
global_stat_net.to(device)
discriminator.to(device)
classifier.to(device)

# 2. Setup Optimizers (The "mechanics" that update the network weights)
# We use Adam optimizers with a learning rate of 1e-4 as per the paper
lr = 1e-4
opt_enc_exp = optim.Adam(encoder_exp.parameters(), lr=lr)
opt_enc_id = optim.Adam(encoder_id.parameters(), lr=lr)
opt_stat = optim.Adam(global_stat_net.parameters(), lr=lr)
opt_disc = optim.Adam(discriminator.parameters(), lr=lr)
opt_cls = optim.Adam(classifier.parameters(), lr=lr)

# 3. Setup Loss Functions
criterion_cls = nn.CrossEntropyLoss() # For emotion classification
criterion_l1 = nn.L1Loss()            # For forcing expression vectors to match
criterion_bce = nn.BCELoss()          # For the True/False discriminator game

# Create a PyTorch DataLoader to feed us batches of 32 images at a time
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

# 4. THE TRAINING LOOP
num_epochs = 30 # We will do 30 epochs as configured

print("\nStarting Training...\n")
for epoch in range(num_epochs):

    total_cls_loss = 0.0
    total_mi_loss = 0.0
    total_disc_loss = 0.0
    correct_preds = 0
    total_samples = 0

    for batch_idx, (img1, img2, label1, _, id1, id2) in enumerate(dataloader):

        # Move images and labels to GPU
        img1, img2, label1 = img1.to(device), img2.to(device), label1.to(device)
        batch_size = img1.size(0)

        # ==========================================
        # PHASE 1: Forward Pass
        # ==========================================
        _, img_feat1, exp_vec1 = encoder_exp(img1)
        _, _, exp_vec2 = encoder_exp(img2)
        _, _, id_vec1 = encoder_id(img1)

        # ==========================================
        # PHASE 2: Train Statistics Network (MINE)
        # ==========================================
        # We want to MAXIMIZE mutual information between the image and its expression vector
        opt_stat.zero_grad()

        # Joint distribution (Real pairs)
        joint_scores = global_stat_net(img_feat1.detach(), exp_vec1.detach())

        # Marginal distribution (Shuffled pairs)
        # We shuffle the expression vectors to break the relationship
        shuffled_indices = torch.randperm(batch_size)
        exp_vec_shuffled = exp_vec1[shuffled_indices]
        marginal_scores = global_stat_net(img_feat1.detach(), exp_vec_shuffled.detach())

        # Donsker-Varadhan Loss Formulation (Numerically stable)
        t_marginal = torch.logsumexp(marginal_scores, dim=0) - math.log(batch_size)
        mi_loss_stat = -(torch.mean(joint_scores) - t_marginal) # Negative because we want to maximize MI

        mi_loss_stat.backward()
        opt_stat.step()

        # ==========================================
        # PHASE 3: Train Discriminator
        # ==========================================
        opt_disc.zero_grad()

        # Real pair (E_M and I_M) -> Should output 1
        real_preds = discriminator(exp_vec1.detach(), id_vec1.detach())
        loss_disc_real = criterion_bce(real_preds, torch.ones(batch_size, 1).to(device))

        # Fake pair (E_M and randomly shuffled I_M) -> Should output 0
        id_vec_shuffled = id_vec1[shuffled_indices]
        fake_preds = discriminator(exp_vec1.detach(), id_vec_shuffled.detach())
        loss_disc_fake = criterion_bce(fake_preds, torch.zeros(batch_size, 1).to(device))

        loss_disc = (loss_disc_real + loss_disc_fake) / 2
        loss_disc.backward()
        opt_disc.step()

        # ==========================================
        # PHASE 4: Train Encoders & Classifier
        # ==========================================
        opt_enc_exp.zero_grad()
        opt_enc_id.zero_grad()
        opt_cls.zero_grad()

        # 1. Classification Loss (Predict the emotion)
        preds = classifier(exp_vec1)
        loss_cls = criterion_cls(preds, label1)

        # Calculate accuracy
        _, predicted = torch.max(preds.data, 1)
        total_samples += label1.size(0)
        correct_preds += (predicted == label1).sum().item()

        # 2. L1 Shared Expression Loss (E_M should equal E_N)
        loss_l1 = criterion_l1(exp_vec1, exp_vec2)

        # 3. MI Maximization Loss (Encoder side)
        joint_scores_enc = global_stat_net(img_feat1, exp_vec1)
        marginal_scores_enc = global_stat_net(img_feat1, exp_vec_shuffled.detach())
        t_marginal_enc = torch.logsumexp(marginal_scores_enc, dim=0) - math.log(batch_size)
        loss_mi_enc = -(torch.mean(joint_scores_enc) - t_marginal_enc)

        # 4. Adversarial Disentanglement Loss (Fool the discriminator into thinking shuffled is real)
        fake_preds_enc = discriminator(exp_vec1, id_vec_shuffled.detach())
        loss_adv = criterion_bce(fake_preds_enc, torch.ones(batch_size, 1).to(device))

        # Combine all losses as per the paper's weighting guidelines (Section 5.3)
        # total_loss = Classification + L1 + MI - Adversarial
        loss_total_enc = loss_cls + (0.1 * loss_l1) + (0.5 * loss_mi_enc) + (0.05 * loss_adv)

        loss_total_enc.backward()
        opt_enc_exp.step()
        opt_enc_id.step()
        opt_cls.step()

        # Tracking metrics
        total_cls_loss += loss_cls.item()
        total_mi_loss += -loss_mi_enc.item() # Track actual MI
        total_disc_loss += loss_disc.item()

    # Print Epoch Summary
    avg_cls_loss = total_cls_loss / len(dataloader)
    avg_mi = total_mi_loss / len(dataloader)
    avg_disc_loss = total_disc_loss / len(dataloader)
    epoch_accuracy = 100.0 * correct_preds / total_samples

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Class Loss: {avg_cls_loss:.4f} | "
          f"Mutual Info: {avg_mi:.4f} | "
          f"Disc Loss: {avg_disc_loss:.4f} | "
          f"Accuracy: {epoch_accuracy:.2f}%")

print("\n Training Complete! The model has successfully learned to decouple Identity from Expression.")

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
import random
from PIL import Image
import torch

# 1. Set the encoders and classifier to evaluation mode (turns off dropout/gradients)
encoder_exp.eval()
encoder_id.eval()
classifier.eval()

print("Extracting features and evaluating accuracy for the entire dataset... (This will take a few seconds)")

all_exp_vecs = []
all_id_vecs = []
all_imgs = []

correct_preds = 0
total_samples = 0

# 2. Pass all images through the network to build a database of vectors
with torch.no_grad():
    for i in range(len(dataset)):
        # Get raw image data and ground truth label
        img_info = dataset.data[i]
        img_raw = Image.open(img_info['path']).convert('L')
        true_label = img_info['emotion_idx']

        # Transform for the model
        img_tensor = transform(img_raw).unsqueeze(0).to(device)

        # Extract the 64-d vectors
        _, _, exp_vec = encoder_exp(img_tensor)
        _, _, id_vec = encoder_id(img_tensor)

        # --- Measure Accuracy ---
        # Pass the expression vector through the classifier
        preds = classifier(exp_vec)
        _, predicted = torch.max(preds.data, 1)

        total_samples += 1
        if predicted.item() == true_label:
            correct_preds += 1
        # ------------------------

        all_exp_vecs.append(exp_vec.cpu())
        all_id_vecs.append(id_vec.cpu())
        all_imgs.append(img_raw)

# Print the Final Numerical Accuracy
final_accuracy = 100.0 * correct_preds / total_samples
print("\n" + "="*50)
print(f"FINAL CLASSIFICATION ACCURACY: {final_accuracy:.2f}%")
print("="*50 + "\n")

# Convert lists to massive tensors for fast math
all_exp_vecs = torch.cat(all_exp_vecs, dim=0)
all_id_vecs = torch.cat(all_id_vecs, dim=0)

# 3. Pick a random query image
query_idx = random.randint(0, len(dataset) - 1)
query_exp = all_exp_vecs[query_idx].unsqueeze(0)
query_id = all_id_vecs[query_idx].unsqueeze(0)

# 4. Calculate Cosine Similarity (How close are the vectors?)
# Closer to 1.0 means they are identical
sim_exp = F.cosine_similarity(query_exp, all_exp_vecs)
sim_id = F.cosine_similarity(query_id, all_id_vecs)

# 5. Get the Top 4 closest matches to avoid running out of identity samples
top_k = 4
_, top_exp_indices = torch.topk(sim_exp, top_k)
_, top_id_indices = torch.topk(sim_id, top_k)

# ==========================================
# 6. Plot the Results!
# ==========================================
fig, axes = plt.subplots(2, top_k, figsize=(15, 6))
fig.suptitle(f"DICE-FER Latent Space Disentanglement Proof\n(Final Model Accuracy: {final_accuracy:.2f}%)", fontsize=16, fontweight='bold')

# Row 1: Expression Retrieval
for i, idx in enumerate(top_exp_indices):
    axes[0, i].imshow(all_imgs[idx.item()], cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title("QUERY IMAGE\n(Look at Emotion)", color='blue', fontweight='bold')
    else:
        axes[0, i].set_title(f"Exp Match {i}\nSim: {sim_exp[idx]:.2f}")

# Row 2: Identity Retrieval
for i, idx in enumerate(top_id_indices):
    axes[1, i].imshow(all_imgs[idx.item()], cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title("QUERY IMAGE\n(Look at Identity)", color='red', fontweight='bold')
    else:
        axes[1, i].set_title(f"ID Match {i}\nSim: {sim_id[idx]:.2f}")

plt.tight_layout()
plt.show()